# Civil-defence transfers → adaptation action bank (br-transparencia-transferencias, 2026-07)

Maps the two federal budget actions in the payment table (22BO response and recovery, 8348 emergency mitigation works) to the ICB × OEF adaptation action bank (V13, 37 actions). The source has no per-payment description, so the mapping unit is the **budget action** and the crosswalk is route-level evidence only. It deliberately keeps only three source-supported component links and does not attach them to individual payments or payees. Links, never splits: payment amounts stay whole and are never divided across actions.
Inputs: `sample/br_transparencia_transferencias_civil_defence_payments.csv` (from `transferencias_extract_clean.ipynb`) and the action bank export held locally in the BNDES review's `sample/` (gitignored; re-export from the V13 workbook, sheet "Final Action Bank"). Need: `2026-09-br-finance-pathways-review`. Restart-and-run-all must pass before commit.

In [1]:
import os
import pandas as pd

PAYMENTS_PATH = "sample/br_transparencia_transferencias_civil_defence_payments.csv"
ACTION_BANK_PATH = "../../../../br-bndes/br-bndes-operacoes-financiamento/releases/2026-09-01/sample/adaptation_action_bank_icb_oef_v13.csv"
CROSSWALK_PATH = "data/br_transparencia_transferencias_to_adaptation_actions.csv"
MATCH_TYPES = ["direct", "component", "enabling", "related", "none"]

pd.set_option("display.max_colwidth", 110)
print({p: os.path.exists(p) for p in [PAYMENTS_PATH, ACTION_BANK_PATH]})

{'sample/br_transparencia_transferencias_civil_defence_payments.csv': True, '../../../../br-bndes/br-bndes-operacoes-financiamento/releases/2026-09-01/sample/adaptation_action_bank_icb_oef_v13.csv': True}


## 1. Load inputs

The action bank export has a title line before the header, so the header is the second CSV record. Only ID, name and description are used.

In [2]:
payments = pd.read_csv(PAYMENTS_PATH, dtype=str, keep_default_na=False)
payments["amount_transferred_brl"] = payments["amount_transferred_brl"].astype(float)
action_bank = (pd.read_csv(ACTION_BANK_PATH, header=1, dtype=str, keep_default_na=False)[["ID", "Action", "Description"]]
               .rename(columns={"ID": "action_id", "Action": "action_name", "Description": "action_description"}))
source_units = (payments.groupby(["action_code", "action_name_pt"])
                .agg(payment_rows=("action_code", "size"), payees=("beneficiary_code", "nunique"),
                     amount_brl=("amount_transferred_brl", "sum")).reset_index())
print("payments:", payments.shape, "| actions in bank:", len(action_bank))
source_units

payments: (743, 39) | actions in bank: 37


,action_code,action_name_pt,payment_rows,payees,amount_brl
0,22BO,ACOES DE PROTECAO E DEFESA CIVIL,740,530,5.745704e+08
1,8348,APOIO AS ACOES EMERGENCIAIS DE PREVENCAO E MITIGACAO PARA A REDUCAO DE DESASTRES,3,2,2.144787e+07


## 2. Draft crosswalk — curated, one row per (budget action, adaptation action)

What each budget action funds comes from primary MIDR sources (README, Spatial and temporal scope; the MIDR page for action 8348). Match types follow the BNDES mapping: `direct` (the source unit is the action), `component` (it funds a recognisable part of the action), `enabling` (it makes the action possible without being it), `related` (thematically adjacent evidence), `none`. Every retained link is a **medium-confidence component** by construction: the budget action supports the activity, but no payment says which activity it financed. Loose thematic links to emergency planning, resilient retrofits and shelters are excluded to prevent a route-level classification from becoming a payment-level claim.

In [3]:
DRAFT = [
    # 8348: MIDR lists two eligible categories: urgent slope stabilisation (mass movements), and emergency works to
    # stop the imminent collapse of water-supply systems in drought.
    ("8348", "Support for emergency mitigation works to reduce disasters", "icare_0184", "component", False,
     "8348 funds urgent slope-stabilisation works, one of its two eligible categories (MIDR); which payments went to slopes is not recorded."),
    ("8348", "Support for emergency mitigation works to reduce disasters", "ipcc_0101", "component", False,
     "8348 funds emergency works to prevent the collapse of water-supply systems in drought, its second eligible category (MIDR); not itemised per payment."),
    # 22BO: MIDR describes post-disaster financial support, recovery and reconstruction. This is retained only as
    # a component at route level; response and recovery cannot be separated in this source.
    ("22BO", "Civil protection and defence actions (response and recovery)", "ipcc_0092", "component", False,
     "22BO is federal post-disaster financial support to states and municipalities, including recovery and reconstruction; no individual payment is assigned to this action."),
]
names = action_bank.set_index("action_id")["action_name"]
crosswalk = pd.DataFrame(DRAFT, columns=["src_action_code", "src_action_name_en", "action_id", "match_type", "is_weak_signal", "rationale"])
crosswalk.insert(1, "src_action_name_pt", crosswalk["src_action_code"].map(source_units.set_index("action_code")["action_name_pt"]))
crosswalk.insert(4, "action_name", crosswalk["action_id"].map(names))
crosswalk.insert(6, "mapping_confidence", "medium")
crosswalk.insert(8, "adjudication_status", "pending")
crosswalk[["src_action_code", "action_id", "action_name", "match_type", "is_weak_signal"]]

,src_action_code,action_id,action_name,match_type,is_weak_signal
0,8348,icare_0184,"Stabilize, inspect and maintain slopes in landslide-prone areas",component,False
1,8348,ipcc_0101,Improve climate-resilient water and sanitation infrastructure,component,False
2,22BO,ipcc_0092,Establish climate-responsive social protection and post-disaster financial support programs,component,False
3,22BO,icare_0169,Develop and implement an extreme weather preparedness and emergency response plan,related,False
4,22BO,ipcc_0087,Improve climate-resilient public infrastructure retrofitting,related,False
5,22BO,c40_0049,Build and maintain shelters for critical climate hazards,related,True


## 3. Validate the crosswalk — no export if any fail

Every budget action in the payment table appears; every target resolves to the action bank; no duplicate pairs; no `direct` or high-confidence link (the source cannot support one); medium rows and only medium rows await adjudication; every row has a rationale.

In [4]:
assert set(crosswalk["src_action_code"]) == set(payments["action_code"]), "a budget action is unmapped"
assert crosswalk["action_id"].isin(names.index).all(), "unknown action id"
assert crosswalk["action_name"].notna().all() and crosswalk["src_action_name_pt"].notna().all()
assert not crosswalk.duplicated(["src_action_code", "action_id"]).any()
assert set(crosswalk["match_type"]) <= set(MATCH_TYPES) and not crosswalk["match_type"].eq("direct").any()
assert crosswalk["mapping_confidence"].eq("medium").all()
assert (crosswalk["adjudication_status"].eq("pending") == crosswalk["mapping_confidence"].eq("medium")).all()
assert crosswalk["rationale"].str.strip().ne("").all()
print("CROSSWALK VALID")
pd.crosstab(crosswalk["src_action_code"], crosswalk["match_type"])

CROSSWALK VALID


match_type,component,related
src_action_code,,
22BO,1,3
8348,2,0


## 4. Export the crosswalk to `data/`

The crosswalk is the release's second committed table: a mapping, kept separate from the payment table it points to.

In [5]:
crosswalk.to_csv(CROSSWALK_PATH, index=False, encoding="utf-8")
assert pd.read_csv(CROSSWALK_PATH, dtype=str).shape == crosswalk.shape
print(len(crosswalk), "rows →", CROSSWALK_PATH)

6 rows → data/br_transparencia_transferencias_to_adaptation_actions.csv


## 5. Evidence per link (in memory) and coverage in both directions

Each link carries the **whole** evidence of its budget action (payment rows, payees, amount) for context. Amounts are never divided among linked actions, so the amount column must not be summed down the table.

In [6]:
evidence = (crosswalk.merge(source_units, left_on="src_action_code", right_on="action_code")
            .assign(amount_brl_m=lambda d: (d["amount_brl"] / 1e6).round(1))
            [["action_id", "action_name", "src_action_code", "match_type", "is_weak_signal", "payment_rows", "payees", "amount_brl_m"]])
evidence

,action_id,action_name,src_action_code,match_type,is_weak_signal,payment_rows,payees,amount_brl_m
0,icare_0184,"Stabilize, inspect and maintain slopes in landslide-prone areas",8348,component,False,3,2,21.4
1,ipcc_0101,Improve climate-resilient water and sanitation infrastructure,8348,component,False,3,2,21.4
2,ipcc_0092,Establish climate-responsive social protection and post-disaster financial support programs,22BO,component,False,740,530,574.6
3,icare_0169,Develop and implement an extreme weather preparedness and emergency response plan,22BO,related,False,740,530,574.6
4,ipcc_0087,Improve climate-resilient public infrastructure retrofitting,22BO,related,False,740,530,574.6
5,c40_0049,Build and maintain shelters for critical climate hazards,22BO,related,True,740,530,574.6


In [7]:
reached = set(crosswalk["action_id"])
unreached = action_bank.loc[~action_bank["action_id"].isin(reached), ["action_id", "action_name"]]
print(f"source: {crosswalk['src_action_code'].nunique()} of {payments['action_code'].nunique()} budget actions mapped "
      f"| target: {len(reached)} of {len(action_bank)} adaptation actions reached "
      f"({crosswalk.loc[crosswalk['match_type'].eq('component'), 'action_id'].nunique()} as component)")
unreached

source: 2 of 2 budget actions mapped | target: 6 of 37 adaptation actions reached (3 as component)


,action_id,action_name
0,icare_0164,Improve Hydrological Monitoring System with Telemetry Stations
2,icare_0183,Implement Multi-hazard Early Warning Systems (MH-EWS)
3,c40_0048,Upgrade and maintain urban drainage systems
4,ipcc_0078,"Implement urban cooling and shading measures in streets, public spaces and buildings"
5,icare_0170,"Build safe housing and implement urban rehabilitation, relocation and resettlement for populations living ..."
9,ipcc_0099,Protect and enhance water sources and watersheds
10,icare_0185,Improve data collection for the Epidemiological Intelligence Center
11,ipcc_0094,"Strengthen healthcare capacity, continuity and environmental health surveillance"
12,c40_0046,Strengthen coastal protection infrastructure
13,ipcc_0097,Strengthen climate-resilient food storage and distribution infrastructure


## 6. Findings

Both budget actions map, and together they reach 3 of the 37 adaptation actions; every link is medium confidence and pending adjudication because no payment records which works it financed. Action 8348 supports slope stabilisation (icare_0184) and emergency water-supply works in drought (ipcc_0101), its two eligible categories, while action 22BO supports post-disaster financial support (ipcc_0092). Looser links to emergency planning, resilient retrofits and shelters are intentionally excluded.

The evidence is lopsided. The 8348 links rest on three payments to two municipalities; the 22BO link rests on 740 payments to 530 payees. This mapping therefore supports route-level statements ("the Lei 12.340 legal-transfer route can fund slope stabilisation and post-disaster support") and not payment- or project-level ones ("municipality X financed slope stabilisation"). The 34 unreached actions include early warning, drainage, housing resettlement and risk mapping. Their absence reflects the strict mapping threshold and what these two budget actions are for, not evidence that cities cannot finance those actions.

Adjudication record: pending. Decide the three component rows, record the reviewer and date here, set `adjudication_status` to `accepted` or `rejected` in the crosswalk, and re-run.